# ****VERTEX NORMAL BILATERAL FILTERING****

In [1]:
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
mesh = o3d.geometry.TriangleMesh.create_sphere(
    radius=1.0,
    resolution=20
)

mesh.compute_vertex_normals()

clean_vertices = np.asarray(mesh.vertices).copy()
triangles = np.asarray(mesh.triangles).copy()

print("Vertices:", len(clean_vertices))
print("Triangles:", len(triangles))

Vertices: 762
Triangles: 1520


In [3]:
o3d.visualization.draw_plotly([mesh])

In [4]:
noise_strength = 0.05

np.random.seed(42)

noise = np.random.normal(
    0,
    noise_strength,
    clean_vertices.shape
)

noisy_vertices = clean_vertices + noise

In [5]:
noisy_mesh = o3d.geometry.TriangleMesh()

noisy_mesh.vertices = o3d.utility.Vector3dVector(
    noisy_vertices
)

noisy_mesh.triangles = o3d.utility.Vector3iVector(
    triangles
)

noisy_mesh.compute_vertex_normals()

TriangleMesh with 762 points and 1520 triangles.

In [6]:
o3d.visualization.draw_plotly([noisy_mesh])

In [7]:
vertices = np.asarray(noisy_mesh.vertices)
normals = np.asarray(noisy_mesh.vertex_normals)
triangles = np.asarray(noisy_mesh.triangles)

neighbors = [set() for _ in range(len(vertices))]

for triangle in triangles:
    a, b, c = triangle

    neighbors[a].add(b)
    neighbors[a].add(c)

    neighbors[b].add(a)
    neighbors[b].add(c)

    neighbors[c].add(a)
    neighbors[c].add(b)

print("Number of vertices:", len(vertices))
print("Neighbors of vertex 100:", sorted(neighbors[100]))

Number of vertices: 762
Neighbors of vertex 100: [np.int32(60), np.int32(61), np.int32(99), np.int32(101), np.int32(139), np.int32(140)]


In [8]:
sigma_s = 0.4
sigma_n = 0.1

print("sigma_s =", sigma_s)
print("sigma_n =", sigma_n)

sigma_s = 0.4
sigma_n = 0.1


In [9]:
new_vertices = vertices.copy()

for i in range(len(vertices)):

    weighted_position = np.zeros(3)
    total_weight = 0.0

    for j in neighbors[i]:

        # -------------------------
        # Spatial distance
        # -------------------------

        distance = np.linalg.norm(
            vertices[i] - vertices[j]
        )

        # -------------------------
        # Spatial weight
        # -------------------------

        spatial_weight = np.exp(
            -(distance ** 2) /
            (2 * sigma_s ** 2)
        )

        # -------------------------
        # Normal similarity
        # -------------------------

        normal_similarity = np.dot(
            normals[i],
            normals[j]
        )

        # -------------------------
        # Normal weight
        # -------------------------

        normal_weight = np.exp(
            -((1 - normal_similarity) ** 2) /
            (2 * sigma_n ** 2)
        )

        # -------------------------
        # Bilateral weight
        # -------------------------

        weight = (
            spatial_weight *
            normal_weight
        )

        # -------------------------
        # Weighted position
        # -------------------------

        weighted_position += (
            weight * vertices[j]
        )

        total_weight += weight

    # -------------------------
    # Normalize
    # -------------------------

    if total_weight > 0:

        new_vertices[i] = (
            weighted_position /
            total_weight
        )

In [10]:
filtered_mesh = o3d.geometry.TriangleMesh()

filtered_mesh.vertices = o3d.utility.Vector3dVector(
    new_vertices
)

filtered_mesh.triangles = o3d.utility.Vector3iVector(
    triangles
)

filtered_mesh.compute_vertex_normals()

print("Filtered mesh created.")

Filtered mesh created.


In [11]:
o3d.visualization.draw_plotly(
    [filtered_mesh]
)

In [12]:
filtered_vertices = np.asarray(
    filtered_mesh.vertices
)

noisy_error = np.linalg.norm(
    noisy_vertices - clean_vertices,
    axis=1
)

filtered_error = np.linalg.norm(
    filtered_vertices - clean_vertices,
    axis=1
)

mean_noisy_error = np.mean(noisy_error)
mean_filtered_error = np.mean(filtered_error)

error_reduction = (
    mean_noisy_error -
    mean_filtered_error
)

percentage_improvement = (
    1 -
    mean_filtered_error /
    mean_noisy_error
) * 100


print("===== DENOISING RESULTS =====")

print(
    "Mean error before filtering:",
    mean_noisy_error
)

print(
    "Mean error after filtering:",
    mean_filtered_error
)

print(
    "Error reduction:",
    error_reduction
)

print(
    "Percentage improvement:",
    percentage_improvement,
    "%"
)

===== DENOISING RESULTS =====
Mean error before filtering: 0.07853260931684036
Mean error after filtering: 0.06595677007502783
Error reduction: 0.012575839241812534
Percentage improvement: 16.013525274673125 %
